In [ ]:
import os
import pandas as pd

In [ ]:
# ========== 需要提取的情景与时间 ==========
ssps = ["ssp1", "ssp2", "ssp3", "ssp4", "ssp5"]
times = [2020, 2040, 2060, 2080, 2100]  # 2020 全 0；其余提取

In [ ]:
# ========== 路径配置（按你给的写法） ==========
resolution = "10km"  # TODO: '0.1km' / '1km' / '3km' / '5km' / '10km' / '20km'
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_base_path = os.path.join(base_path, "data")

path_name = f"Points_China_all_{resolution}"
print(path_name)

# ✅ 输出目录：base_path/output/path_name
out_dir = os.path.join(base_path, "output", path_name)
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)

In [ ]:
def read_excel_cell_H15_as_percent_str(csv_path: str) -> str:
    """
    对齐 Excel 的 $H$15：
    - 用 header=None 读入，保证“第1行就是Excel第1行”
    - H列为第8列 -> 0-based 列索引=7
    - 15行为第15行 -> 0-based 行索引=14
    Excel公式：0.01 * H15 -> 得到比例；再格式化成百分号字符串
    """
    df = pd.read_csv(csv_path, header=None)
    raw = float(df.iat[14, 7])            # H15
    ratio = 0.01 * raw                    # Excel: 0.01 * H15
    return f"{ratio * 100:.2f}%"          # 显示成 10.45%

# ========== 构建结果表 ==========
result = pd.DataFrame({"Time": times})
for ssp in ssps:
    result[ssp] = "0.00%"  # 先默认全 0（包括 2020）

for t in [2040, 2060, 2080, 2100]:
    ssp_time = str(t)
    for ssp in ssps:
        fname = f"province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_{ssp}_{ssp_time}.csv"
        in_path = os.path.join(out_dir, ssp, ssp_time, fname)

        if not os.path.exists(in_path):
            raise FileNotFoundError(f"找不到文件：{in_path}")

        result.loc[result["Time"] == t, ssp] = read_excel_cell_H15_as_percent_str(in_path)

# ========== 保存 ==========
out_csv = os.path.join(out_dir, "risk_pre_ssp_ssp_time.csv")

# 如果你希望输出长得像示例（制表符分隔），用 sep="\t"
result.to_csv(out_csv, index=False, sep="\t", encoding="utf-8-sig")

print(f"已保存：{out_csv}")
print(result)

绘图

In [ ]:
# ========== Heatmap + 边际均值条形（TNR=9；无colorbar；v=0~37；边际柱同一cmap；边框灰；所有文字黑；所有刻度线灰；右侧y轴无刻度线）==========
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker  # ✅ 新增：用于彻底移除刻度

ssp_cols = ["ssp1", "ssp2", "ssp3", "ssp4", "ssp5"]

# --- 把 "10.45%" -> 10.45（数值，单位：%）---
plot_df = result.copy()
for c in ssp_cols:
    plot_df[c] = plot_df[c].astype(str).str.replace("%", "", regex=False).astype(float)

times = plot_df["Time"].astype(int).tolist()
Z = plot_df[ssp_cols].to_numpy().T  # (SSP, Time)
ssp_labels = [c.upper() for c in ssp_cols]

mean_by_time = plot_df[ssp_cols].mean(axis=1).to_numpy()  # (Time,)
mean_by_ssp  = plot_df[ssp_cols].mean(axis=0).to_numpy()  # (SSP,)

# --- 统一颜色规范 ---
SPINE_GRAY = "#7A7A7A"   # 边框
TICK_GRAY  = "#7A7A7A"   # 刻度线
TEXT_BLACK = "black"     # 所有文字黑色

# --- 风格：Times New Roman + 字体 9（文字默认黑色） ---
mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 9,
    "text.color": TEXT_BLACK,
    "axes.labelcolor": TEXT_BLACK,
    "xtick.color": TEXT_BLACK,   # 刻度文字黑
    "ytick.color": TEXT_BLACK,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# --- 颜色：值域固定 0~37% ---
cmap = LinearSegmentedColormap.from_list("peach_to_blue", ["#FFF4EE", "#FB7D5D"], N=256)
vmin, vmax = 0.0, 37.0
norm = Normalize(vmin=vmin, vmax=vmax)

# 边际柱颜色：按均值映射到同一 cmap/norm
top_colors   = cmap(norm(mean_by_time))
right_colors = cmap(norm(mean_by_ssp))

# --- 版面：上方均值柱 + 主热力图 + 右侧均值条 ---
fig = plt.figure(figsize=(3.35, 2.45), dpi=300)
gs = GridSpec(
    2, 2, figure=fig,
    width_ratios=[5.0, 1.4],
    height_ratios=[1.2, 5.0],
    wspace=0.15, hspace=0.12
)

ax_top   = fig.add_subplot(gs[0, 0])
ax_main  = fig.add_subplot(gs[1, 0])
ax_right = fig.add_subplot(gs[1, 1])   # ✅ 不再 sharey，避免影响主图
ax_blank = fig.add_subplot(gs[0, 1]); ax_blank.axis("off")

# --- 主热力图 ---
ax_main.imshow(
    Z, cmap=cmap, norm=norm,
    aspect="auto", origin="upper", interpolation="nearest"
)

xpos = np.arange(len(times))
ypos = np.arange(len(ssp_labels))

ax_main.set_xticks(xpos)
ax_main.set_xticklabels([str(t) for t in times], color=TEXT_BLACK)
ax_main.set_yticks(ypos)
ax_main.set_yticklabels(ssp_labels, color=TEXT_BLACK)  # ✅ SSP1-5 保证显示

ax_main.set_xlabel("Year", color=TEXT_BLACK)
ax_main.set_ylabel("Scenario", color=TEXT_BLACK)

# 细网格分隔（表格感）
ax_main.set_xticks(np.arange(-.5, len(times), 1), minor=True)
ax_main.set_yticks(np.arange(-.5, len(ssp_labels), 1), minor=True)
ax_main.grid(which="minor", linewidth=0.6, alpha=0.35)
ax_main.tick_params(which="minor", bottom=False, left=False)

# 单元格标注：全部黑色
for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        val = float(Z[i, j])
        ax_main.text(j, i, f"{val:.2f}%", ha="center", va="center", color=TEXT_BLACK, fontsize=8)

# 主图边框：四边灰色
for side in ["top", "right", "bottom", "left"]:
    ax_main.spines[side].set_visible(True)
    ax_main.spines[side].set_linewidth(0.8)
    ax_main.spines[side].set_color(SPINE_GRAY)

# --- 上方：各年份均值（颜色映射；y轴刻度=0/10/20；y=0 横轴线；x轴无刻度线） ---
ax_top.bar(xpos, mean_by_time, width=0.7, edgecolor="none", color=top_colors)
ax_top.set_xlim(-0.5, len(times) - 0.5)
ax_top.set_ylabel("Mean (%)", color=TEXT_BLACK)

ax_top.set_xticks(xpos)
ax_top.tick_params(axis="x", labelbottom=False, length=0, color=TICK_GRAY, labelcolor=TEXT_BLACK)

ax_top.set_yticks([0, 10, 20])
ax_top.set_ylim(0, 20)

ax_top.spines["bottom"].set_visible(True)
ax_top.spines["bottom"].set_linewidth(0.8)
ax_top.spines["bottom"].set_position(("data", 0))
ax_top.spines["bottom"].set_color(SPINE_GRAY)

ax_top.spines["top"].set_visible(False)
ax_top.spines["right"].set_visible(False)
ax_top.spines["left"].set_visible(True)
ax_top.spines["left"].set_linewidth(0.8)
ax_top.spines["left"].set_color(SPINE_GRAY)

ax_top.grid(True, axis="y", linewidth=0.5, alpha=0.25)

# --- 右侧：各SSP跨年份均值（颜色映射；x轴刻度=0/10/20；彻底移除y轴刻度线/标签） ---
ax_right.barh(ypos, mean_by_ssp, height=0.7, edgecolor="none", color=right_colors)
ax_right.set_xlabel("Mean (%)", color=TEXT_BLACK)

# ✅ 对齐主图y轴范围（含反向），保证条形与热力图行严格对齐
ax_right.set_ylim(ax_main.get_ylim())

# ✅ 彻底移除右侧小图 y 轴刻度线与标签
ax_right.set_yticks([])
ax_right.tick_params(axis="y", which="both", left=False, right=False, labelleft=False, labelright=False, length=0)

ax_right.set_xticks([0, 10, 20])
ax_right.set_xlim(0, 20)
ax_right.tick_params(axis="x", which="both", color=TICK_GRAY, labelcolor=TEXT_BLACK)

# 右侧小图边框（left/bottom灰色；top/right不显示）
ax_right.spines["top"].set_visible(False)
ax_right.spines["right"].set_visible(False)
ax_right.spines["left"].set_visible(True)
ax_right.spines["bottom"].set_visible(True)
ax_right.spines["left"].set_linewidth(0.8)
ax_right.spines["bottom"].set_linewidth(0.8)
ax_right.spines["left"].set_color(SPINE_GRAY)
ax_right.spines["bottom"].set_color(SPINE_GRAY)

ax_right.grid(True, axis="x", linewidth=0.5, alpha=0.25)

# --- 所有刻度线灰色；文字黑色（主图/上图/右图一致） ---
for ax in [ax_top, ax_main, ax_right]:
    ax.tick_params(axis="both", which="major", color=TICK_GRAY, labelcolor=TEXT_BLACK)
    ax.tick_params(axis="both", which="minor", color=TICK_GRAY, labelcolor=TEXT_BLACK)


# --- 只保存为 SVG（不保存 PNG/PDF）---
fig_path_svg = os.path.join(out_dir, "risk_pre_ssp_ssp_time_b.svg")
plt.savefig(fig_path_svg, format="svg", bbox_inches="tight")
plt.show()
plt.close(fig)

print("Figure saved:")
print(fig_path_svg)



绘制趋势图

In [ ]:
# ========== 绘图：Nature 风格（简洁、出版级）+ 不确定性误差带（随时间累积）==========
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy.interpolate import UnivariateSpline
from matplotlib.ticker import MultipleLocator
from matplotlib import font_manager

# 如果你这里是从文件读回来的，就取消注释：
# result = pd.read_csv(out_csv, sep="\t")

# --- 把 "10.45%" -> 10.45（数值，单位：%）---
plot_df = result.copy()
ssp_cols = ["ssp1", "ssp2", "ssp3", "ssp4", "ssp5"]
for c in ssp_cols:
    plot_df[c] = plot_df[c].astype(str).str.replace("%", "", regex=False).astype(float)

x = plot_df["Time"].values.astype(float)

# ===== 尺寸：8.6 cm × 6.35 cm -> inch =====
cm_to_inch = 1 / 2.54
fig_w = 8.6 * cm_to_inch
fig_h = 6.35 * cm_to_inch

# ===== 字体：尽量锁定 Times New Roman（无则 fallback 到 Times/DejaVu Serif）=====
tnr = font_manager.FontProperties(family="Times New Roman")

# ===== 样式：字号9；字体全黑；坐标轴灰色 =====
axis_gray = "#7a7a7a"
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 9,

    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "legend.frameon": False,

    "axes.linewidth": 0.8,
    "axes.edgecolor": axis_gray,

    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.direction": "out",
    "ytick.direction": "out",

    "svg.fonttype": "path",
})

fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=300)

# # ===== 颜色渐变：#c7d4d9 -> #E5A694 =====
# start = np.array(to_rgb("#c7d4d9"))
# end   = np.array(to_rgb("#E29A86"))
# colors = [tuple(start + (end - start) * i / (len(ssp_cols) - 1)) for i in range(len(ssp_cols))]

# ===== 5个SSP分别指定颜色（替换掉原来的渐变代码）=====
ssp_cols = ["ssp1", "ssp2", "ssp3", "ssp4", "ssp5"]

# ssp_color = {
#     "ssp1": "#DC3739AA",  # green
#     "ssp2": "#8694b8",  # olive/green
#     "ssp3": "#9570b3",  # purple
#     "ssp4": "#FB7D5D",  # yellow/orange
#     "ssp5": "#b44f02",  # orange/red
# }

ssp_color = {
    "ssp1": "#8694b8",  # green
    "ssp2": "#9570b3",  # olive/green
    "ssp3": "#FB7D5D",  # purple
    "ssp4": "#b44f02",  # yellow/orange
    "ssp5": "#DC3739AA",  # orange/red
}

# 如果你后面仍然用 colors[i] 的写法，也可以保留这个列表：
colors = [ssp_color[c] for c in ssp_cols]


# 为样条拟合准备：x 必须严格递增
order = np.argsort(x)
x_sorted = x[order]
x_dense = np.linspace(x_sorted.min(), x_sorted.max(), 300)

# ===== 平滑参数 =====
s_val = 50
print(f"Smoothing factor s = {s_val}")

# ================== 不确定性设置（你可按需要改这三个参数） ==================
# 误差随时间累积：2020(起始年)=0，2100(末年)=err_max_2100
y_all = plot_df[ssp_cols].to_numpy(dtype=float)
data_range = np.nanmax(y_all) - np.nanmin(y_all)

err_max_2100 = max(1.0, 0.15 * data_range)  # 2100年的最大误差（单位：百分点），默认取数据范围的10%，且至少1
err_power = 1.0                              # 增长形状：1=线性；>1 后期增长更快；<1 前期增长更快
band_alpha = 0.3                            # 误差带透明度
# ======================================================================

# 把 x_dense 归一化到 [0,1]，保证起点误差=0，终点误差=err_max_2100
t_norm = (x_dense - x_sorted.min()) / (x_sorted.max() - x_sorted.min())
t_norm = np.clip(t_norm, 0, 1)
err_dense = err_max_2100 * (t_norm ** err_power)  # 随年份增加而增大

for i, ssp in enumerate(ssp_cols):
    color = ssp_color[ssp]  # <- 新增这一行
    y = plot_df[ssp].values.astype(float)[order]
    spline = UnivariateSpline(x_sorted, y, k=3, s=s_val)
    y_dense = spline(x_dense)

    # ===== 不确定性误差带（随时间累积）=====
    y_lower = np.maximum(0, y_dense - err_dense)  # 防止下界小于0（你若允许负值可去掉这行的maximum）
    y_upper = y_dense + err_dense
    ax.fill_between(
        x_dense, y_lower, y_upper,
        # color=colors[i],
        color=color,
        alpha=band_alpha,
        linewidth=0,
        zorder=1
    )

    # ===== 趋势线（画在误差带上方）=====
    ax.plot(
        x_dense, y_dense,
        # color=colors[i],
        color=color,
        linewidth=1.2,
        linestyle="-",
        label=ssp.upper(),
        zorder=2
    )

# 轴标签
ax.set_xlabel("Year", fontproperties=tnr, color="black")
ax.set_ylabel("Risk probability change (%)", fontproperties=tnr, color="black")

# x刻度：显示原始年份
ax.set_xticks(x_sorted)

# y轴：0-50%，每10%一个刻度
ax.set_ylim(0, 50)
ax.yaxis.set_major_locator(MultipleLocator(10))

# 网格
ax.grid(True, axis="y", linewidth=0.5, alpha=0.25)
ax.grid(False, axis="x")

# 四周边框都显示，并设为灰色
for side in ["left", "bottom", "top", "right"]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_color(axis_gray)

# tick：文字黑色；刻度线灰色
ax.tick_params(axis="both", labelcolor="black", color=axis_gray)

# 强制 tick label 使用 Times New Roman（如果系统可用）
for lab in (ax.get_xticklabels() + ax.get_yticklabels()):
    lab.set_fontproperties(tnr)
    lab.set_color("black")

# 图例
leg = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    bbox_transform=ax.transAxes,
    ncol=3,
    handlelength=2.0,
    columnspacing=1.0,
    borderaxespad=0.2,
    prop=tnr
)
for t in leg.get_texts():
    t.set_color("black")

fig.tight_layout()

# 输出
fig_path_svg  = os.path.join(out_dir, "risk_pre_ssp_ssp_time_a_ppt_safe_with_uncertainty.svg")
plt.savefig(fig_path_svg, bbox_inches="tight")

plt.show()
plt.close(fig)

print("Figure saved:")
print(fig_path_svg)


In [ ]:
# ================= Figure 3 (right panel only): Province mean baseline vs future =================
# It reads: province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_{ssp}_{year}.csv
# and outputs (6cm x 6cm): risk_pre_ssp_ssp_time_c_province_{ssp}_{year}.svg
import os
# Example: change these two to draw other scenarios / times
ssp = 'ssp4'
ssp_time = '2100'
fname = f'province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_{ssp}_{ssp_time}.csv'
in_path = os.path.join(out_dir, ssp, ssp_time, fname)
from fig3_province_scatter_plot import main as fig3_main
fig3_main(in_path=in_path, out_dir=out_dir, ssp=ssp, year=ssp_time, width_cm=6, height_cm=6)


In [ ]:
# ================= Figure 3 (right panel only): Province mean baseline vs future =================
# It reads: province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_{ssp}_{year}.csv
# and outputs (6cm x 6cm): risk_pre_ssp_ssp_time_c_province_{ssp}_{year}.svg
import os
from fig3_province_scatter_plot_updated import main_batch as fig3_batch

# 5个情景 + 年份
cases_cfg = [
    ("ssp1", "2040"),
    ("ssp2", "2100"),
    ("ssp3", "2080"),
    ("ssp4", "2100"),
    ("ssp5", "2100"),
]

# 组装每个情景的输入文件路径
case_specs = []
for ssp, ssp_time in cases_cfg:
    fname = f"province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_{ssp}_{ssp_time}.csv"
    in_path = os.path.join(out_dir, ssp, ssp_time, fname)
    case_specs.append((ssp, ssp_time, in_path))

# 一次性画 5 张 + 1 张平均
fig_paths = fig3_batch(
    cases=case_specs,
    out_dir=out_dir,
    width_cm=6,
    height_cm=6,
    make_average=True,
    average_ssp="mean",
    average_year="mean_5cases",
    # average_subdir="mean_fig3"  # 可选：想固定输出到 out_dir/mean_fig3/ 就打开
    # save_violin=False
)

print("All figures saved:")
for p in fig_paths:
    print(p)

In [ ]:
# ================= Figure 3 (right panel only): Province mean baseline vs future =================
# It reads: province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_{ssp}_{year}.csv
# and outputs (6cm x 6cm): risk_pre_ssp_ssp_time_c_province_{ssp}_{year}.svg
import os
from fig3_province_scatter_plot_updated_with_violin_v5 import main_batch as fig3_batch

# 5个情景 + 年份
cases_cfg = [
    ("ssp1", "2040"),
    ("ssp2", "2100"),
    ("ssp3", "2080"),
    ("ssp4", "2100"),
    ("ssp5", "2100"),
]

# 组装每个情景的输入文件路径
case_specs = []
for ssp, ssp_time in cases_cfg:
    fname = f"province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_{ssp}_{ssp_time}.csv"
    in_path = os.path.join(out_dir, ssp, ssp_time, fname)
    case_specs.append((ssp, ssp_time, in_path))

# 一次性画 5 张 + 1 张平均
fig_paths = fig3_batch(
    cases=case_specs,
    out_dir=out_dir,
    width_cm=6,
    height_cm=6,
    make_average=True,
    average_ssp="mean",
    average_year="mean_5cases",
    # average_subdir="mean_fig3"  # 可选：想固定输出到 out_dir/mean_fig3/ 就打开
    # save_violin=False
)

print("All figures saved:")
for p in fig_paths:
    print(p)